# compare_models_cv.ipynb

Notebook: comparação de modelos (Ridge, Lasso, RandomForest) com GridSearchCV e validação cruzada (k=5).
Gera `models_comparison.csv` e `predicoes_test.csv` como artefatos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Config
RANDOM_STATE = 42
CV = 5

# -------------------------
# Robust CSV discovery and encoding handling (BLOCO SUBSTITUÍDO)
# -------------------------
repo_root = Path('.')
csv_path = None

def normalize_filename(s):
    s = s.strip()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return s.replace(' ', '_').lower()

# lista todos os CSVs no top-level
csvs = sorted(repo_root.glob('*.csv'))
if csvs:
    matches = []
    for p in csvs:
        if 'preco' in normalize_filename(p.name):
            matches.append((normalize_filename(p.name), p))
    if matches:
        matches.sort()
        csv_path = matches[0][1]
    else:
        csv_path = csvs[0]

if csv_path is None:
    raise FileNotFoundError('Nenhum arquivo CSV encontrado no repositório. Adicione Precos_de_casas.csv ou similar.')

print('Usando arquivo CSV:', csv_path)

# tenta ler com alguns encodings comuns
for enc in ('utf-8', 'latin1', 'cp1252'):
    try:
        df = pd.read_csv(csv_path, encoding=enc)
        print(f'Read CSV with encoding: {enc}')
        break
    except Exception as e:
        last_exc = e
else:
    raise last_exc
# -------------------------
# Fim do bloco substituído
# -------------------------

# Normalizar colunas
def normalize_col(c):
    return c.strip().replace(' ', '_').replace('ç','c').replace('ã','a').replace('é','e').replace('ó','o').replace('í','i')

df.columns = [normalize_col(c) for c in df.columns]

# identificar target
target_candidates = [c for c in df.columns if 'preco' in c.lower()]
if not target_candidates:
    raise ValueError('Coluna alvo nao encontrada')
ycol = target_candidates[-1]

# remover linhas sem target
df = df.dropna(subset=[ycol])

# separar X/y
X = df.drop(columns=[ycol])
y = df[ycol].astype(float)

# identificar tipos
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()

# pipeline
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
# Force dense output from OneHotEncoder to avoid sparse solver issues with Ridge on the runner
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))])
preproc = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

results = []

# Models + reduced param grids (smaller for faster execution)
models_and_params = [
    ('Ridge', Ridge(random_state=RANDOM_STATE, solver='cholesky'), {'model__alpha': [1.0]}),
    ('Lasso', Lasso(random_state=RANDOM_STATE, max_iter=10000), {'model__alpha': [0.01]}),
    ('RandomForest', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), {'model__n_estimators': [100], 'model__max_depth': [10]})
]

for name, estimator, param_grid in models_and_params:
    print(f'Running GridSearch for {name} with grid: {param_grid}')
    pipe = Pipeline([('preproc', preproc), ('model', estimator)])
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=CV, scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    # cross-validate metrics on training data with the best estimator
    from sklearn.model_selection import cross_validate
    scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
    cv_res = cross_validate(best, X_train, y_train, cv=CV, scoring=scoring, n_jobs=-1)
    mean_rmse = -np.mean(cv_res['test_rmse'])
    std_rmse = np.std(cv_res['test_rmse'])
    mean_mae = -np.mean(cv_res['test_mae'])
    std_mae = np.std(cv_res['test_mae'])
    mean_r2 = np.mean(cv_res['test_r2'])
    std_r2 = np.std(cv_res['test_r2'])
    # test set performance
    y_pred = best.predict(X_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)
    results.append({
        'model': name,
        'mean_cv_rmse': mean_rmse,
        'std_cv_rmse': std_rmse,
        'mean_cv_mae': mean_mae,
        'std_cv_mae': std_mae,
        'mean_cv_r2': mean_r2,
        'std_cv_r2': std_r2,
        'test_rmse': test_rmse,
        'test_mae': test_mae,
        'test_r2': test_r2,
        'best_params': gs.best_params_
    })
    # salvar previsoes para este modelo
    preds_df = pd.DataFrame({'y_true': y_test.values, f'y_pred_{name}': y_pred}, index=y_test.index)
    preds_path = repo_root / f'predicoes_test_{name}.csv'
    preds_df.to_csv(preds_path, index=False)
    print(f'Finished {name}: test RMSE={test_rmse:.2f}, test R2={test_r2:.4f}')

# consolidar resultados e salvar
results_df = pd.DataFrame(results).sort_values('test_rmse')
results_df.to_csv('models_comparison.csv', index=False)
print('Saved models_comparison.csv')

# salvar previsoes combinadas (unir por index)
pred_files = list(repo_root.glob('predicoes_test_*.csv'))
if pred_files:
    combined = None
    for pf in pred_files:
        d = pd.read_csv(pf)
        if combined is None:
            combined = d
        else:
            combined = combined.merge(d, left_index=True, right_index=True, how='outer')
    combined.to_csv('predicoes_test.csv', index=False)
    print('Saved predicoes_test.csv')